# 7. Performance: diagnose before you tune

The code of this module's lessons in [Learn Apache Spark by Doing](https://dataenglab.com/courses/learn-apache-spark-by-doing/), in lesson order, on Apache Spark 4.2.0 in local mode: one machine, no cluster.

Run the cells from top to bottom. Under a cell, the result the lesson shows is there to compare with yours.

Not available in Colab: the Spark UI (its port is not reachable from a notebook). Where a lesson sends you to it, run that part on your machine; here, `df.explain()` prints the plan.

In [ ]:
# Setup: Java 17+, PySpark 4.2.0 and the webshop data. About a minute the first time.
import os, re, subprocess, sys, urllib.request, zipfile

PYSPARK_VERSION = "4.2.0"
DATA_URL = 'https://github.com/dataeng-lab/learn-apache-spark-by-doing/raw/v1/datasets.zip'

if "google.colab" in sys.modules:
    version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    major = re.search(r'version "(?:1\.)?(\d+)', version)
    if not major or int(major.group(1)) < 17:       # PySpark 4 does not start on Java 8 or 11
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                       shell=True, check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"], check=True)

if not os.path.isdir("datasets"):
    if DATA_URL is None:
        raise RuntimeError("datasets/ is missing: upload the course's datasets folder in the Files panel")
    urllib.request.urlretrieve(DATA_URL, "datasets.zip")
    zipfile.ZipFile("datasets.zip").extractall(".")

print("datasets:", sorted(os.listdir("datasets")))

**Starting point.** The DataFrames and names the earlier modules built. Not part of the lesson text.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("webshop")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

from pyspark.sql.types import (
    DecimalType, IntegerType, StringType, StructField, StructType, TimestampType,
)

ORDERS_SCHEMA = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=True),
    StructField("order_ts", TimestampType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("amount", DecimalType(10, 2), nullable=True),
    StructField("currency", StringType(), nullable=True),
    StructField("updated_at", TimestampType(), nullable=True),
])
orders_schema = ORDERS_SCHEMA   # some lessons use this name

orders = spark.read.csv("datasets/orders.csv", header=True, schema=ORDERS_SCHEMA)

customers = spark.read.csv("datasets/customers.csv", header=True, inferSchema=True)

## A method for a slow job

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/a-method-for-a-slow-job/)

## What to look for in the Spark UI

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/what-to-look-for-in-the-spark-ui/)

Not run here: writes the event log to S3; the Spark UI and the History Server are not reachable from Colab.

```python
spark = (
    SparkSession.builder.appName("webshop-nightly")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "s3://webshop-logs/spark-events/")
    .getOrCreate()
)
```

## Data skew: when one task holds everyone up

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/data-skew/)

The lesson saves this as `measure_skew.py`.

In [ ]:
from pyspark.sql import functions as F

enriched = orders.join(customers, on="customer_id").select("order_id", "amount", "country")

sizes = (
    enriched.repartition(8, "country")
    .withColumn("pid", F.spark_partition_id())
    .groupBy("pid")
    .agg(F.count("*").alias("rows"))
    .orderBy(F.col("rows").desc())
)
sizes.show()

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
rows per partition after shuffling on `country`:
  partition  4:  3345  #######################################################
  partition  3:   417  ######
  partition  0:   215  ###
  partition  6:    69  #
  partition  5:    65  #
  partition  1:    41  #

largest partition is 82x the smallest.
One task processes the big one. The stage is not done until that task is done.
```

## Broadcast joins

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/broadcast-joins/)

In [ ]:
from pyspark.sql import functions as F

enriched = orders.join(F.broadcast(customers), on="customer_id")

## What AQE adapts at runtime

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/adaptive-query-execution/)

## cache and persist: a trade, not a speed-up

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/cache-and-persist/)

Not run here: `expensive_query()` and `...` stand for your own job.

```python
from pyspark import StorageLevel

daily = expensive_query()
daily.persist(StorageLevel.MEMORY_ONLY)   # drop partitions that do not fit; recompute them
daily.count()                             # nothing is cached until an action materialises it
...
daily.unpersist()                         # release it; this one takes effect immediately
```